### Different Models access using Langchain

In [17]:
import os
os.environ['GROQ_API_KEY'] = os.getenv('GROQ_API_KEY')
from langchain.chat_models import init_chat_model
model = init_chat_model('groq:llama-3.3-70b-versatile')
response = model.invoke('Hello what is the whether in hyderabad')
response.content


"Hello! Hyderabad, being a city located in the southern part of India, has a tropical wet and dry climate. The weather in Hyderabad can vary depending on the time of year. Here's a general overview of the typical weather conditions in Hyderabad:\n\n**Summer (March to May)**: Hot and dry, with temperatures often reaching 40°C (104°F) or higher.\n\n**Monsoon (June to September)**: Warm and humid, with frequent rainfall and temperatures ranging from 25°C to 35°C (77°F to 95°F).\n\n**Winter (October to February)**: Mild and pleasant, with temperatures ranging from 15°C to 25°C (59°F to 77°F).\n\nAs I'm an AI, I don't have real-time access to current weather conditions. However, I can suggest some ways for you to find out the current weather in Hyderabad:\n\n1. Check online weather websites like AccuWeather, Weather.com, or the Indian Meteorological Department's website.\n2. Download a weather app on your smartphone, such as Dark Sky or Weather Underground.\n3. Tune into local news or radio

In [18]:
response = model.invoke('Yes i would like to know more about the whether in Hyd..')
response.content

"Hyderabad, the capital city of Telangana, India, has a unique climate. The weather in Hyderabad is characterized as a semi-arid climate, with hot summers and mild winters. Here's a breakdown of the typical weather conditions in Hyderabad:\n\n**Seasons:**\n\n1. **Summer (March to May)**: This is the hottest season in Hyderabad, with temperatures often reaching 40°C (104°F) or more. The summer months are dry, with very little rainfall.\n2. **Monsoon (June to September)**: The southwest monsoon brings relief from the heat, with moderate to heavy rainfall. The temperatures cool down, and the humidity increases.\n3. **Winter (December to February)**: Winters in Hyderabad are mild, with average temperatures ranging from 12°C (54°F) to 25°C (77°F). This is the coolest season, with minimal rainfall.\n4. **Autumn (October to November)**: The post-monsoon season, also known as autumn, is characterized by warm temperatures and low humidity.\n\n**Temperature:**\n\n* Average high temperature: 32°C

In [19]:
os.environ['GOOGLE_API_KEY'] = os.getenv('GOOGLE_API_KEY')
gemini_model = init_chat_model('gemini-3.5-flash',model_provider="google_genai")
response_gemini = gemini_model.invoke('hey how are you my name is anil')
response_gemini.text

"Hello Anil! It's great to meet you. I'm doing really well, thank you for asking! \n\nHow are you doing today? How can I help you?"

In [20]:
import langchain
langchain.__version__

'1.3.13'

### otherways of integrating model without chat model

In [21]:
###accessing openai using package without chat_model
from langchain_openai import ChatOpenAI
model = ChatOpenAI(model='gpt-5.6')
resp = model.invoke('how are you openAI')
resp.content

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

In [ ]:
###accessing grok using package without chat_model
from langchain_groq import ChatGroq
model = ChatGroq(model='llama-3.3-70b-versatile')
resp = model.invoke('how are you Groq')
resp.content

"I'm just a language model, I don't have feelings or emotions like humans do, but I'm functioning properly and ready to assist you. However, I'm curious - who or what is Groq? Is that a character or a reference I should be familiar with?"

In [ ]:
###accessing gemini using package without chat_model
from langchain_google_genai import ChatGoogleGenerativeAI
model = ChatGoogleGenerativeAI(model = 'gemini-3.5-flash')
resp = model.invoke('How are you gemini')
resp.content[0]['text']

"Hello! I'm doing well, thank you for asking. How are you doing today? How can I help you?"

## Batch and streaming

In [ ]:
for chunk in model.stream('How are you gemini') :
    print(chunk.text, end = '|', flush=True)


Hello! I'|m doing really well, thank you for asking. How are you doing today? 

How can I help you?|||

In [ ]:
for chunk in model.batch(['hello what is my name?','how are you doing','do you have emotions']):
    print(chunk.content)    

In [ ]:
##making different kind of tools
import os
from langchain.chat_models import init_chat_model
from langchain.tools import tool
os.environ['GROQ_API_KEY'] = os.getenv('GROQ_API_KEY')

@tool
def check_whether (city:str)->str :
    '''give the whether of a city'''
    return f'whether in {city} is cloudy and sunny...'

new_gemini_model = init_chat_model(model='groq:llama-3.3-70b-versatile')
model_with_tool = new_gemini_model.bind_tools([check_whether])
resp = model_with_tool.invoke('what is the whether like in Hyderabad?')
# print(resp)
for chunk in resp.tool_calls:
    print(f'Tool : {chunk['name']}' )
    print(f'Args: {chunk['args']}')
    print(f'ID: {chunk['id']}')

Tool : check_whether
Args: {'city': 'Hyderabad'}
ID: x8dkj158t


In [ ]:
##Tools execution Loops
message = [{"role":"user","content":"what is the whether in Boston?"}]
ai_msg = model_with_tool.invoke(message)
message.append(ai_msg)

for tool_call in ai_msg.tool_calls:
   tool_resp = check_whether.invoke(tool_call)
   message.append(tool_resp)

final_resp = model_with_tool.invoke(message) 
final_resp.content

'I only provide the function call to get the weather. The actual weather in Boston is provided by the function call result, which is cloudy and sunny.'

In [26]:
import os
from langchain.messages import SystemMessage,HumanMessage,AIMessage
from langchain_google_genai import ChatGoogleGenerativeAI
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")
s_message = SystemMessage('You are Programmer with 30 Years of Experience in AI')
ai_message = AIMessage('How can i help you ?')
h_message = HumanMessage('explain about rag in 10 lines')
model = ChatGoogleGenerativeAI(model = 'gemini-3.5-flash')

res = model.invoke([s_message,ai_message,h_message])
res

AIMessage(content=[{'type': 'text', 'text': 'Here is RAG explained in exactly 10 lines, from my perspective as an AI veteran:\n\n1. **The Definition:** RAG (Retrieval-Augmented Generation) is an architecture that connects LLMs to external, real-time, or private data sources.\n2. **The Problem It Solves:** It fixes LLM "knowledge cutoffs" and hallucinations without the massive expense of retraining or fine-tuning.\n3. **Step 1 (Ingestion):** Your proprietary documents are broken into chunks and converted into mathematical vectors (embeddings).\n4. **Step 2 (Storage):** These vectors are stored in a specialized Vector Database (like Pinecone, Milvus, or Chroma) for fast semantic search.\n5. **Step 3 (Retrieval):** When a user asks a question, the system searches the database for chunks with the most similar vector meaning.\n6. **Step 4 (Augmentation):** The system injects these relevant document chunks directly into the prompt alongside the user\'s original query.\n7. **Step 5 (Generatio

In [ ]:
from langchain.messages import SystemMessage,AIMessage,HumanMessage,ToolMessage
h_msg = HumanMessage('what is the whether in fransisco')

ai_msg = AIMessage(content=[],tool_calls=[{"name":"check_whether","args":{"location":"fransisco"},"id":"anil_123" }])

tool_msg = ToolMessage(content='70 degress and sunny..',tool_call_id = 'anil_123')

messages = [h_msg,ai_msg,tool_msg]
resp = model.invoke(messages)
resp.content

"The current weather in San Francisco is 70 degrees and sunny. San Francisco is known for its mild climate, with temperatures ranging from the mid-50s to the mid-70s (13°C to 24°C) throughout the year. However, the weather can vary depending on the time of year and other factors.\n\nIf you're planning a trip to San Francisco, here's a rough idea of what you can expect:\n\n* Summer (June to August): Warm and sunny, with temperatures in the mid-70s to mid-80s (24°C to 30°C)\n* Fall (September to November): Mild and pleasant, with temperatures in the mid-60s to mid-70s (18°C to 24°C)\n* Winter (December to February): Cool and rainy, with temperatures in the mid-50s to mid-60s (13°C to 18°C)\n* Spring (March to May): Mild and sunny, with temperatures in the mid-60s to mid-70s (18°C to 24°C)\n\nKeep in mind that these are general temperature ranges, and the weather can vary from year to year. It's always a good idea to check the forecast before your trip to get a more accurate idea of the w

In [ ]:
## model generating the structured out with pydantic
from pydantic import BaseModel,Field

##class defines the structure of output
class Movie (BaseModel):
    title:str = Field('title of the movie')
    year:int = Field('year of release of movie')
    ratings:float = Field('ratings of the movie')
    director:str = Field('director of the movie')

model_with_structured_output = model.with_structured_output(Movie)##mapped structure class with model
response = model_with_structured_output.invoke('Bahubali')
response

Movie(title='Baahubali', year=2015, ratings=8.1, director='S. S. Rajamouli')

In [ ]:
## model generating the structured out with pydantic
from pydantic import BaseModel,Field

##structed as well as row included output
class Movie (BaseModel):
    title:str = Field(..., description='title of the movie')
    year:int = Field(...,description='year of release of movie')
    ratings:float = Field(...,description='ratings of the movie')
    director:str = Field(...,description='director of the movie')

model_with_structured_output = model.with_structured_output(Movie,include_raw=True)##mapped structure class with model
response = model_with_structured_output.invoke('Bahubali')
response["parsed"]

Movie(title='Bahubali', year=2015, ratings=8.1, director='S.S. Rajamouli')

In [ ]:
## model generating the structured out with pydantic
from pydantic import BaseModel,Field

##Nested structured output
class Actor (BaseModel):
    name:str
    role:str
    age:int
    birth_place:str

class MovieDetails (BaseModel):
    title:str = Field('Movie Name')
    year:str = Field('Year of realease')
    cast:list[Actor]
    genre:list[str]
    budget:float | None = Field(None, description='budget of the Movie')

model_with_structured_output = model.with_structured_output(MovieDetails)##mapped structure class with model
response = model_with_structured_output.invoke('i want to know the details of movie bahubali')
response

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

In [34]:
###typeDict
from typing_extensions import TypedDict,Annotated
import os
from langchain.messages import SystemMessage,HumanMessage,AIMessage

class Actor (TypedDict):
    name:Annotated[str, ..., 'name of actor']
    role:Annotated[str, ..., 'role of actor']
    age:Annotated[int, ..., 'age of actor']
    birth_place:Annotated[str, ..., 'birth place of actor']

class MovieDetails (TypedDict):
    title: Annotated[str, ..., 'title of the movie']
    cast: Annotated[list[Actor], ..., 'actor details']
    rating: Annotated[int, ..., 'rating of the movie']
    budget: Annotated[str, ..., 'budget of the movie']

model_with_structured_output_typeDict = model.with_structured_output(MovieDetails)
response = model_with_structured_output_typeDict.invoke('i want to know details of movie vikramarkudu')
response['cast'][0]['role']

'ACP Vikram Singh Rathore / Athili Sathi Babu'

In [61]:
### tools response in a structed way
import os
from pydantic import BaseModel,Field
from langchain.agents import create_agent
from langchain_google_genai import ChatGoogleGenerativeAI
os.environ['GOOGLE_API_KEY'] = os.getenv("GOOGLE_API_KEY")
model_genai = ChatGoogleGenerativeAI(model = "gemini-3.5-flash")

class Movie (BaseModel):
    title:str = Field(..., description='title of the movie')
    year:int = Field(...,description='year of release of movie')
    ratings:float = Field(...,description='ratings of the movie')
    director:str = Field(...,description='director of the movie')

agent = create_agent(model= model_genai, response_format=Movie)

resp = agent.invoke({"messages": [{"role":"user","content":"extract details from this information Bahubali,2001,9.8,ss rajamouli"}]})

resp

{'messages': [HumanMessage(content='extract details from this information Bahubali,2001,9.8,ss rajamouli', additional_kwargs={}, response_metadata={}, id='76319685-a34e-4916-abee-5a5546d44991'),
  AIMessage(content=[{'type': 'text', 'text': '{"title":"Bahubali","year":2001,"ratings":9.8,"director":"ss rajamouli"}', 'extras': {'signature': 'EqYJCqMJARFNMg8nzXgJR8Xt6UtlZUt8mSyitqdG0YLVjbNQCi1NL0cFPMDadNZDV3u+ERmz7P9XL7nlDMSpuoneoLCDalUgM/msfWqc9XALAKqi2UfRWWTL+RRzKnf7mA0YWE0lnvFcfu7KncJTbG0kxsah1Ybw5raNA7elu4tE/IHboyPhjAtwi/cIVUbW0N2NlTkXEqGqIs8gIOWluB9rZucX7GCvdUft/BJJ1pAjHalW4nvHGk1yX3MgrJIEv5AqSwP9voZu+UCA82UlMhAM/KXBZ/CQStH22ATQV/wW003DaqQ44XOhVOx/FiZC8IlWsr1iqUp3BI+Cfb1l+CiejOi3XDacHJSJGkg6d9DoXlsdFbhSNBeRwsImulGAhHKtw4fWvRkNoFvTnf+WMkXpo+4EZfRhcKkKUHPQNOsiEGC+GxKrXJX5OZJMKVn05I97MMj3kv2PXmynOinQdxxv6X59719d7mjxGEc9fmlB0k788z6641re20Jr9J91Cc3o3X4Tey3U50O+BKbXZQNYM9DZxj+C0tIibnb9AqPkxaqqCV+iFGA4ces728rZm9M3NqGnqikEXfmzAG0UAEnzjVRHEOXsSwL/3ZxgymLdScnvTE9Gjq5rmFnh7TmMB3hsZ6qEj2Z2+ABq7d